---
## GSM8K SFT — Supervised Fine-Tuning on Real Math Problems

We now go beyond the small synthetic dataset and fine-tune on the full GSM8K training set.
Each GSM8K example already contains a step-by-step solution; we simply reformat it as
`<think>chain-of-thought</think><answer>final_answer</answer>` and train the model to
produce this exact structure.

The key differences from the earlier random-format SFT:
- **Real reasoning traces** from GSM8K rather than synthetic arithmetic steps
- **Full model fine-tuning** (all parameters, not just embeddings)
- **`SFTTrainer`** class that mirrors the `GRPOTrainer` API for consistency

In [ ]:
from utils.models import load_model, load_tokenizer

model_name = "Qwen/Qwen3-1.7B-Base"
model = load_model(model_name)
tokenizer = load_tokenizer(model_name)

In [ ]:
def generate_prompt(question):
    """
    Wraps a question into the DeepSeek-R1 prompt format.
    """
    prompt = (
        "A conversation between User and Assistant. The user asks a question, and the Assistant solves it. "
        "The assistant first thinks about the reasoning process in the mind and then provides the user with the answer. "
        "The reasoning process and answer are enclosed within <think>...</think> and <answer>...</answer> tags, "
        "respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>. "
        f"User: {question} Assistant: "
    )
    return prompt


### 1. Build the GSM8K SFT dataset

`GSM8KSFTDataset` tokenizes each example as:
- **prompt** (labels = -100) → the question wrapped in the DeepSeek-R1 template
- **completion** (labels = token ids) → `<think>{cot}</think><answer>{answer}</answer>`

Examples longer than `max_length` tokens are dropped to keep memory usage predictable.

In [ ]:
from data.gsm8k import GSM8KSFTDataset

train_sft = GSM8KSFTDataset(
    tokenizer=tokenizer,
    prompt_template=generate_prompt,
    split="train",
    max_length=512,
)

# Inspect one example
sample = train_sft[0]
full_text = tokenizer.decode(sample["input_ids"])
print("--- Full tokenized example ---")
print(full_text)
print(f"\nInput length : {sample['input_ids'].shape[0]} tokens")
print(f"Label -100s  : {(sample['labels'] == -100).sum().item()} (prompt tokens masked)")

### 2. Train with `SFTTrainer`

`SFTTrainer` handles:
- Padding batches to the same length
- Standard cross-entropy loss on completion tokens only
- Periodic evaluation (loss on a held-out subset)
- Saving the best checkpoint

In [ ]:
from training.sft import SFTTrainer
from training.configs import SFTConfig

config = SFTConfig(
    lr=2e-5,
    epochs=3,
    batch_size=8,
    max_length=512,
    grad_clip=1.0,
    eval_every=200,
    eval_samples=100,
)

# Optional: load a separate eval split (raw GSM8KSFTDataset on the test set)
eval_sft = GSM8KSFTDataset(
    tokenizer=tokenizer,
    prompt_template=generate_prompt,
    split="test",
    max_length=512,
)

trainer = SFTTrainer(model=model, tokenizer=tokenizer, config=config)
trainer.train(train_sft, eval_dataset=eval_sft, run_name="Qwen/Qwen3-1.7B-Base-gsm8k-sft")

### 3. Save and evaluate

After training we save the model and run the format quality evaluation
used earlier in the notebook to confirm the model now reliably produces
`<think>` / `<answer>` blocks.

In [ ]:
from utils.models import save_model

save_model(model, tokenizer, "Qwen/Qwen3-1.7B-Base-gsm8k-sft")

In [ ]:
from utils import checks

def generate_responses(model, question, num_responses, temperature):
    inputs = tokenizer(generate_prompt(question), return_tensors="pt").to(model.device)
    input_len = inputs.input_ids.shape[1]

    outputs = model.generate(
                    **inputs, 
                    max_new_tokens=256,    # High, but just a demonstration
                    do_sample=True,         # Enable sampling
                    temperature=temperature,        # Adds randomness
                    num_return_sequences=num_responses, # This does the parallel work for n tries
                    top_p=0.9,
                    pad_token_id=tokenizer.eos_token_id
                )

    output_texts = tokenizer.batch_decode(outputs[:, input_len:], skip_special_tokens=True)
    return output_texts

def evaluate(model, questions, answers=[], G=1, temperature = 1.0):
    num_questions = len(questions)

    answer_count = 0
    think_count = 0
    perfect_count = 0
    correct_count = 0

    for q_idx in range(num_questions):
        question = questions[q_idx]
        responses = generate_responses(model, question, G, temperature)
        for raw_response in responses:
            if checks.has_complete_answer_block(raw_response): answer_count += 1
            if checks.has_complete_thinking_block(raw_response): think_count += 1
            if checks.is_format_correct(raw_response): perfect_count += 1
            if answers:
                ground_truth = answers[q_idx]
                if checks.is_correct_answer(raw_response, ground_truth): correct_count += 1

    num_samples = num_questions * G
    print(f"Thinking: {(think_count/num_samples)*100:.1f}% of tries.\n",
        f"Answer: {(answer_count/num_samples)*100:.1f}% of tries\n",
        f"Perfect format: {(perfect_count/num_samples)*100:.1f}% of tries")
    if answers:
        print(f"Correct answers: {(correct_count/num_samples)*100:.1f}% of tries")
    
questions = ["What is 11 * 9?", 
             "Sam has 4 apples, and eats 2 of them, and gives one to Anna, how many apples does he have left?",
             "Solve: (5 + 1) * 21"]
evaluate(model=model, questions=questions, G=8)

In [ ]:
# Check that the model now uses the think/answer format consistently
questions = [
    "What is 11 * 9?",
    "Sam has 4 apples, eats 2, and gives 1 to Anna. How many does he have left?",
    "Solve: (5 + 1) * 21",
    "What is the next number in the sequence [1, 2, 4, 8, 16]?",
]
answers = ["99", "1", "126", "32"]
evaluate(model=model, questions=questions, answers=answers, G=16, temperature=1.0)

---
## Evaluate SFT model on GSM8K test set

Runs the model on the full GSM8K test set (or a subset), then plots:
1. **Cumulative accuracy** — how accuracy stabilises as more questions are evaluated
2. **Think token distribution** — histogram of how many tokens the model spends reasoning

In [ ]:
from eval.eval_gsm8k import evaluate_sft

metrics, records = evaluate_sft(
    model=model,
    tokenizer=tokenizer,
    split="test",
    n_examples=100,        # set to None for full test set (~1319 examples)
    batch_size=8,
    temperature=0.3,       # low temperature for deterministic eval
    max_new_tokens=512,
    helper="<think>",
    prompt_style="deepseek",
    show_examples=3,       # print this many example Q/A pairs
)

print(metrics)

### Breakdown by outcome

Quick summary of correct / wrong / failed-to-extract counts.

In [ ]:
correct   = sum(r['correct'] for r in records)
wrong     = sum(not r['correct'] and r['pred'] is not None for r in records)
no_answer = sum(r['pred'] is None for r in records)
total     = len(records)

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(
    ["Correct", "Wrong", "No extract"],
    [correct, wrong, no_answer],
    color=["#4caf50", "#f44336", "#9e9e9e"],
    edgecolor="white",
    width=0.5,
)
for bar, val in zip(bars, [correct, wrong, no_answer]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f"{val}\n({val/total:.1%})", ha='center', va='bottom', fontsize=10)
ax.set_ylabel("Count")
ax.set_title(f"GSM8K test — {total} examples")
ax.set_ylim(0, total * 1.15)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()